In [1]:
import pandas as pd 
import numpy as np
import os 
import cv2
from tqdm import tqdm
import h5py
import hdf5storage
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

In [2]:
# functions 
def reconstruct_movie(spatial_weights, temporal_weights, sample_start, sample_end):
    """
    Reconstructs a movie by scaling each neuron's spatial footprint by its
    temporal weight over time.
    
    Parameters
    ----------
    spatial_weights : np.ndarray
        Array of shape (H, W, N) with spatial footprint of each neuron.
    temporal_weights : np.ndarray
        Array of shape (T, N) with temporal activity of each neuron.
    sample_start : int
        Starting sample index (inclusive).
    sample_end : int
        Ending sample index (exclusive).
    
    Returns
    -------
    movie : np.ndarray
        Reconstructed movie of shape (T_slice, H, W), where T_slice = sample_end - sample_start.
    """
    # Extract the temporal slice
    t_slice = temporal_weights[sample_start:sample_end]  # shape (T_slice, N)
    # Reconstruct movie: tensordot over neuron axis
    movie = np.tensordot(t_slice, spatial_weights, axes=([1], [2]))  # shape (T_slice, H, W)
    return movie

def display_movie_animation(movie, sample_start=0, interval=100, cmap='gray'):
    """
    Display a reconstructed movie using matplotlib FuncAnimation in Jupyter.
    
    Parameters
    ----------
    movie : np.ndarray
        Array of shape (T, H, W) representing the movie frames.
    sample_start : int
        The starting sample index for labeling frames.
    interval : int
        Delay between frames in milliseconds.
    cmap : str
        Colormap for displaying frames.
    """
    T, H, W = movie.shape
    fig, ax = plt.subplots(figsize=(6, 6))
    im = ax.imshow(movie[0], cmap=cmap, aspect='equal')
    ax.set_title(f'Frame {sample_start}')
    ax.axis('off')

    def update(frame_idx):
        im.set_array(movie[frame_idx])
        ax.set_title(f'Frame {sample_start + frame_idx}')
        return [im]

    ani = FuncAnimation(fig, update, frames=T, interval=interval, blit=True)
    plt.close(fig)  # Prevents double display
    
    return HTML(ani.to_jshtml())

def save_reconstructed_avi(movie, output_file, fps):
    """
    Save a 3D movie array (T, H, W) as a grayscale AVI that FIJI can open.

    Parameters
    ----------
    movie : np.ndarray
        Array of shape (T, H, W), one frame per timepoint.
    output_file : str
        Path to the .avi you want to write.
    fps : float
        Frames per second for the output video.
    """
    T, H, W = movie.shape
    # Define codec & create VideoWriter (isColor=False for single‐channel)
    fourcc = cv2.VideoWriter_fourcc(*'MJPG')
    out    = cv2.VideoWriter(output_file, fourcc, fps, (W, H), isColor=False)

    for frame in movie:
        # normalize each frame to 0–255 → uint8
        frame_uint8 = cv2.normalize(
            frame, None, 0, 255, cv2.NORM_MINMAX
        ).astype('uint8')
        out.write(frame_uint8)

    out.release()
    print(f"Saved AVI to {output_file!r}")

def extract_video_subset(input_path, output_path, start_frame, end_frame, codec='MJPG'):
    """
    Extract frames [start_frame, end_frame) from `input_path` and save to `output_path`.
    Uses MJPG codec by default for FIJI compatibility.

    Parameters
    ----------
    input_path : str
        Path to the original behavior video (e.g., 'video_output.avi').
    output_path : str
        Path where the subset video will be saved (e.g., 'video_output_subset.avi').
    start_frame : int
        Index of the first frame to include (inclusive, zero-based).
    end_frame : int
        Index of the last frame to include (exclusive).
    codec : str
        FourCC code for the output video. Default 'MJPG' for FIJI.
    """
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open video file {input_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc = cv2.VideoWriter_fourcc(*codec)
    out = cv2.VideoWriter(output_path, fourcc, fps, (w, h))

    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret or frame_idx >= end_frame:
            break
        if frame_idx >= start_frame:
            out.write(frame)
        frame_idx += 1

    cap.release()
    out.release()
    print(f"Saved frames {start_frame} to {end_frame-1} to {output_path}")

import cv2
import numpy as np

def save_concatenated_vertically(
    calcium_movie,         # np.ndarray of shape (T, Hc, Wc)
    behavior_subset_path,  # str: path to behavior subset AVI
    combined_output_path,  # str: where to save concatenated AVI
    fps=15,                # float: frames per second for output
    alpha=0.5,             # float: transparency for calcium color mapping
    colormap=cv2.COLORMAP_JET
):
    """
    Concatenate behavior and calcium movies vertically. The wider frame
    determines the output width; the narrower is padded horizontally.

    Parameters
    ----------
    calcium_movie : np.ndarray
        Array of shape (T, Hc, Wc) representing calcium intensities.
    behavior_subset_path : str
        Path to the behavior video subset AVI (T frames, Hb, Wb).
    combined_output_path : str
        Output path for the concatenated AVI.
    fps : float
        Frames per second for the output.
    alpha : float
        Weight for color mapping (unused here but kept for consistency).
    colormap : int
        OpenCV colormap for calcium overlay (here used to convert to BGR).
    """
    # Open behavior subset
    cap = cv2.VideoCapture(behavior_subset_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open behavior video: {behavior_subset_path}")

    # Get behavior dimensions and frame count
    Hb = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    Wb = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    fps_behavior = cap.get(cv2.CAP_PROP_FPS)
    n_frames = calcium_movie.shape[0]

    # Calcium dimensions
    Hc, Wc = calcium_movie.shape[1], calcium_movie.shape[2]

    # Determine output dimensions
    final_width = max(Wb, Wc)
    final_height = Hb + Hc

    # Setup VideoWriter for MJPG (FIJI-compatible)
    fourcc = cv2.VideoWriter_fourcc(*'MJPG')
    out = cv2.VideoWriter(combined_output_path, fourcc, fps, (final_width, final_height))

    for idx in range(n_frames):
        ret, beh_frame = cap.read()
        if not ret:
            break

        # Process behavior frame: pad horizontally if needed
        if Wb < final_width:
            pad_left = (final_width - Wb) // 2
            pad_right = final_width - Wb - pad_left
            beh_frame_padded = cv2.copyMakeBorder(
                beh_frame,
                top=0, bottom=0, left=pad_left, right=pad_right,
                borderType=cv2.BORDER_CONSTANT, value=[0,0,0]
            )
        else:
            beh_frame_padded = beh_frame

        # Process calcium frame: normalize → uint8 → apply colormap → pad
        ca = calcium_movie[idx]
        ca_norm = cv2.normalize(ca, None, 0, 255, cv2.NORM_MINMAX).astype('uint8')
        ca_color = cv2.applyColorMap(ca_norm, colormap)

        if Wc < final_width:
            pad_left = (final_width - Wc) // 2
            pad_right = final_width - Wc - pad_left
            ca_color_padded = cv2.copyMakeBorder(
                ca_color,
                top=0, bottom=0, left=pad_left, right=pad_right,
                borderType=cv2.BORDER_CONSTANT, value=[0,0,0]
            )
        else:
            ca_color_padded = ca_color

        # Vertically concatenate: behavior on top, calcium below
        combined_frame = cv2.vconcat([beh_frame_padded, ca_color_padded])

        # Write to output
        out.write(combined_frame)

    cap.release()
    out.release()
    print(f"Saved vertically concatenated movie to: {combined_output_path}")


import cv2
import numpy as np

def save_concatenated_vertically_grayscale(
    calcium_movie,         # np.ndarray of shape (T, Hc, Wc), float or double
    behavior_subset_path,  # str: path to behavior subset AVI
    combined_output_path,  # str: where to save the grayscale AVI
    fps=15                 # float: frames per second for output
):
    """
    Concatenate behavior and calcium movies *vertically* into a single‐channel
    (grayscale) AVI. The output width is the max of the two widths; the narrower
    video is zero‐padded horizontally. Behavior frames (assumed color) are
    converted to gray; calcium frames are normalized to 0–255 uint8.

    Parameters
    ----------
    calcium_movie : np.ndarray
        Array of shape (T, Hc, Wc) representing calcium intensities.
    behavior_subset_path : str
        Path to the behavior video subset AVI (T frames, Hb×Wb color).
    combined_output_path : str
        Output path for the concatenated grayscale AVI.
    fps : float
        Frames per second for the output.
    """
    cap = cv2.VideoCapture(behavior_subset_path)
    if not cap.isOpened():
        raise IOError(f"Cannot open behavior video: {behavior_subset_path}")

    # Grab behavior properties
    Hb = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    Wb = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    fps_behavior = cap.get(cv2.CAP_PROP_FPS)
    T = calcium_movie.shape[0]

    # Calcium dimensions
    Hc, Wc = calcium_movie.shape[1], calcium_movie.shape[2]

    # Final output dimensions
    final_width = max(Wb, Wc)
    final_height = Hb + Hc

    # Create a VideoWriter for grayscale AVI (isColor=False)
    fourcc = cv2.VideoWriter_fourcc(*'MJPG')
    out = cv2.VideoWriter(combined_output_path, fourcc, fps, (final_width, final_height), isColor=False)

    frame_idx = 0
    while frame_idx < T:
        ret, beh_frame = cap.read()
        if not ret:
            break

        # 1) Convert behavior frame to grayscale
        beh_gray = cv2.cvtColor(beh_frame, cv2.COLOR_BGR2GRAY)  # (Hb, Wb)

        # 2) Normalize calcium frame to 0–255 and cast to uint8
        ca = calcium_movie[frame_idx]
        ca_gray = cv2.normalize(ca, None, 0, 255, cv2.NORM_MINMAX).astype('uint8')  # (Hc, Wc)

        # 3) Pad behavior gray horizontally if needed
        if Wb < final_width:
            pad_left = (final_width - Wb) // 2
            pad_right = final_width - Wb - pad_left
            beh_padded = cv2.copyMakeBorder(
                beh_gray,
                top=0, bottom=0, left=pad_left, right=pad_right,
                borderType=cv2.BORDER_CONSTANT, value=0
            )
        else:
            beh_padded = beh_gray

        # 4) Pad calcium gray horizontally if needed
        if Wc < final_width:
            pad_left = (final_width - Wc) // 2
            pad_right = final_width - Wc - pad_left
            ca_padded = cv2.copyMakeBorder(
                ca_gray,
                top=0, bottom=0, left=pad_left, right=pad_right,
                borderType=cv2.BORDER_CONSTANT, value=0
            )
        else:
            ca_padded = ca_gray

        # 5) Stack vertically: behavior on top, calcium below
        combined_frame = cv2.vconcat([beh_padded, ca_padded])  # shape (final_height, final_width)

        # 6) Write as a single-channel frame
        out.write(combined_frame)

        frame_idx += 1

    cap.release()
    out.release()
    print(f"Saved grayscale vertically concatenated movie to: {combined_output_path}")


In [7]:
#path to eZTrack data 
ezTrackLocations = r'/Users/johnmarshall/Documents/Analysis/miniscope_analysis/miniscopeLinearTrack/Part_4_Day1rec_DMSO_11/Mouse11Day1_custom_cropped_output_LocationOutput.csv'
ezTrack = alignedTraces = pd.read_csv(ezTrackLocations)

#path to aligned eZTrack data
dirPath = '/Users/johnmarshall/Documents/Analysis/miniscope_analysis/miniscopeLinearTrack/Part_4_Day1rec_DMSO_11/'
alignedFile = '1_37_motion_correctedcellTracesAlignedToTracking.csv'

#path to aligned tracking with velocity 
#alignedFileWithVelocity = 'alignedTracesWithVelocity.csv'

#load calcium signal aligned to tracking data 
alignedTraces = pd.read_csv(dirPath+alignedFile)
#alignedFileWithVelocity = pd.read_csv(dirPath+alignedFileWithVelocity)

#extract "good" cells from aligned traces data 
cell_cols = alignedTraces.filter(regex=r'^cell_').columns
cell_ids = [int(col.split('_')[1]) for col in cell_cols]
/

()

In [8]:
alignedTraces

,Unnamed: 0,cell_12,cell_13,cell_14,cell_15,cell_19,cell_21,cell_22,cell_25,cell_27,...,cell_1314,cell_1315,cell_1317,Frame Number,Time Stamp (ms),Buffer Index,closestBehavCamFrameIdx,X_coor,Y_coor,Distance_px
0,0 days 00:00:00,0.661929,0.413001,1.494183,-0.766036,-0.649710,1.740848,1.557957,1.099901,-0.516615,...,-0.665207,1.701197,-0.900212,0,-21,0,0,1.000000,17.000000,0.000000
1,0 days 00:00:00.050000,0.755292,-0.171724,1.502397,-0.804833,-0.578913,0.702669,0.871224,0.749012,-1.251515,...,-0.475032,0.931724,-1.672080,1,32,0,1,1.000000,18.961089,1.961089
2,0 days 00:00:00.100000,0.209402,-0.077480,1.217756,-1.435824,-0.455961,1.279007,1.067169,-0.128330,1.128362,...,-0.578195,1.940201,-0.455581,2,80,0,2,1.000000,18.959677,0.001412
3,0 days 00:00:00.150000,0.526674,0.032346,1.614694,-0.098954,-0.607497,0.436783,1.197700,0.339635,-0.039178,...,-0.047258,1.008860,-0.630109,3,131,0,2,1.000000,18.959677,0.001412
4,0 days 00:00:00.200000,0.459089,-0.321650,1.060506,-1.443203,-0.238626,1.110664,0.806882,0.256595,0.268090,...,-0.109851,1.154502,-0.166875,4,182,0,3,1.000000,19.000000,0.040323
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36005,0 days 00:30:00.250000,-0.113662,-0.183885,2.704232,-0.686337,2.702309,-1.275400,0.425866,-0.983263,-0.419880,...,-0.177226,0.464474,-0.236355,36005,1824080,0,26150,14.353785,12.324779,0.300490
36006,0 days 00:30:00.300000,0.496993,0.183803,2.817280,-0.086100,1.593365,-1.343665,0.255834,-0.948490,-1.401832,...,-0.588805,0.165595,-0.015984,36006,1824131,0,26150,14.353785,12.324779,0.300490
36007,0 days 00:30:00.350000,-0.718442,-0.011438,3.052339,-0.167741,2.722773,-1.151372,-0.057891,-0.856564,-0.928276,...,-0.122325,1.053783,-0.669589,36007,1824181,0,26151,15.563084,12.077882,1.234246
36008,0 days 00:30:00.400000,0.581232,0.216738,2.865067,0.430805,1.769982,-1.065549,0.112603,-0.886966,-0.139182,...,0.157474,0.077088,0.367224,36008,1824232,0,26152,14.183544,12.114608,1.380029


In [9]:
alignedTracesLocationData = alignedTraces[['closestBehavCamFrameIdx', 'X_coor', 'Y_coor']]
aligned_calcium_signals = (
    alignedTraces
      .filter(regex=r'^cell_')                            # all the cell_* columns
      .assign(closestBehavCamFrameIdx = 
              alignedTraces['closestBehavCamFrameIdx'])  # add the index column
)
locationReIndexedToBehaviorData = (
    alignedTracesLocationData
      .groupby('closestBehavCamFrameIdx')           # group by frame-idx
      .mean(numeric_only=True)                     # mean of only number-dtype cols
)
calciumTracesReIndexedToBehaviorData = (
    aligned_calcium_signals
      .groupby('closestBehavCamFrameIdx')           # group by frame-idx
      .mean(numeric_only=True)                     # mean of only number-dtype cols
)

In [ ]:
# matlab data - spatial footprints and calcium signals 
mat = hdf5storage.loadmat(dirPath + '1_37_motion_corrected.mat')

In [24]:
output = mat['output']
fields = output.dtype.names
print(fields)
# unwrap the single record
record = output[0]
spatial_weights  = record['spatial_weights']
temporal_weights = record['temporal_weights']
# filter out good cells from ActSort 
spatial_weights_filtered = spatial_weights[:, :, cell_ids]
temporal_weights_filtered = temporal_weights[:, cell_ids]
np.shape(spatial_weights)

('spatial_weights', 'temporal_weights', 'info', 'config')


(600, 600, 301)

In [25]:
# reconstruct intensity movie 
sample_start = 0
totalSamplesAlignedToBehavior = len(calciumTracesReIndexedToBehaviorData.values)
sample_end = 100
#sample_end = totalSamplesAlignedToBehavior
calcium_movie = reconstruct_movie(spatial_weights_filtered, calciumTracesReIndexedToBehaviorData.values, sample_start, sample_end)
save_reconstructed_avi(movie, dirPath+'CalciumTracesReconstructedFromEXTRACTdataSubset.avi', fps=15)

Saved AVI to '/Users/johnmarshall/Documents/Analysis/miniscope_analysis/miniscopeLinearTrack/2025.4/m328/2025_04_10_328_15_59_58_b2/CalciumTracesReconstructedFromEXTRACTdataSubset.avi'


In [26]:
# reconstructed calcium signal movie joined to behavior movie 
pathToBehaviorVideo = dirPath+'My_WebCam/'+'video_output.avi' 
subset_path = dirPath + 'My_WebCam/video_output_subset.avi'
extract_video_subset(pathToBehaviorVideo, subset_path, sample_start, sample_end)
combined_output_path = dirPath + 'combinedVideo.avi'

#save_concatenated_vertically(calcium_movie, subset_path, combined_output_path, fps=15)
save_concatenated_vertically_grayscale(calcium_movie, subset_path, combined_output_path, fps=15)

Saved frames 0 to 99 to /Users/johnmarshall/Documents/Analysis/miniscope_analysis/miniscopeLinearTrack/2025.4/m328/2025_04_10_328_15_59_58_b2/My_WebCam/video_output_subset.avi
Saved grayscale vertically concatenated movie to: /Users/johnmarshall/Documents/Analysis/miniscope_analysis/miniscopeLinearTrack/2025.4/m328/2025_04_10_328_15_59_58_b2/combinedVideo.avi


In [27]:
calciumTracesReIndexedToBehaviorData

,cell_0,cell_1,cell_2,cell_3,cell_4,cell_7,cell_9,cell_10,cell_11,cell_17,...,cell_249,cell_255,cell_257,cell_265,cell_266,cell_274,cell_279,cell_286,cell_296,cell_300
closestBehavCamFrameIdx,,,,,,,,,,,,,,,,,,,,,
0,-0.517713,4.484764,0.381960,0.035891,0.036944,1.935868,-0.035760,-0.352228,1.407909,3.475072,...,1.381332,2.105667,0.822662,0.721554,0.302304,0.611756,0.230086,-0.605095,2.334729,2.163733
1,-0.484956,3.632884,-0.441250,-0.177387,0.284546,2.055512,-0.390445,0.554764,2.076181,3.293647,...,1.190611,2.006899,0.805629,0.788808,-0.157268,0.560434,0.143564,-0.205992,2.486964,0.507958
2,-0.524364,3.102902,0.217553,-0.629499,0.099012,1.506088,-0.131508,-0.414635,1.791406,2.655635,...,0.607764,2.271527,-0.358154,0.753029,0.841154,-0.088475,0.429381,-1.405403,2.012349,1.417181
3,-0.318655,2.934727,0.642404,-0.370112,0.233681,1.155710,-0.567853,0.311860,1.680241,2.549960,...,1.172224,2.082390,-0.161544,0.813552,-0.147732,-0.324766,-0.023387,-1.084980,2.089265,1.143603
4,-0.474101,2.806412,-0.799658,0.115840,-0.308727,1.588002,-0.322479,-0.354427,0.792684,1.634344,...,0.565844,2.211419,-0.341018,0.824307,-0.284734,-0.025275,-0.078923,-0.792351,2.315628,1.116790
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20565,0.256742,-0.073933,-0.390584,0.311036,-0.222495,0.255596,-0.822022,-0.874950,-1.296963,-1.001454,...,0.837525,1.051105,-0.852265,0.641868,-0.036676,0.329841,0.628700,1.090245,-0.687992,-1.133813
20566,0.259669,0.278993,-0.225285,0.120499,-0.179442,-0.173513,-0.177403,-0.435590,0.016893,-0.744624,...,0.191094,0.513491,-1.068962,0.180824,-1.456588,-0.792724,0.009064,1.256137,-0.414299,-1.861119
20567,-0.381514,-0.038495,-0.732041,0.102051,-0.709249,-0.365920,-0.041849,-1.004916,0.839608,-0.949134,...,0.569684,-0.114259,-0.808005,0.755851,-1.141002,0.183167,0.093739,0.857895,0.596439,-1.208960


In [28]:
display_movie_animation(movie, sample_start=0, interval=100)

In [29]:
##validate with video overlay  
#small video 

# ── CONFIG ─────────────────────────────────────────────────────────────────────

dirPath      = '/Users/johnmarshall/Documents/Analysis/miniscope_analysis/miniscopeLinearTrack/2025.4/m328/2025_04_10_328_15_59_58_b2/My_WebCam/'
rotatedVideo = 'video_output.avi'
input_vid    = os.path.join(dirPath, rotatedVideo)
output_vid   = os.path.join(dirPath, 'with_velocity_overlay.mp4')

# your velocity array (length == # frames)
vel_array = locationReIndexedToBehaviorData['velocity2dSpatialFiltered'].to_numpy()

# your per-behavior-frame X positions (make sure it's indexed by frame 0…n-1)
# e.g. binned = binned.reset_index().set_index('frame')
x_pos = locationReIndexedToBehaviorData['X_coor'].to_numpy()

# height of the black bar you want to add
bar_h = 50

# ── SET UP I/O ─────────────────────────────────────────────────────────────────

cap      = cv2.VideoCapture(input_vid)
fps      = cap.get(cv2.CAP_PROP_FPS)
w        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h        = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# choose a codec that FIJI likes for AVIs
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out    = cv2.VideoWriter(output_vid, fourcc, fps, (w, h + bar_h))

# text & dot styling
font        = cv2.FONT_HERSHEY_SIMPLEX
font_scale  = 1
thickness   = 2
margin      = 10
dot_radius  = 5

# ── PROCESS FRAMES ──────────────────────────────────────────────────────────────

for frame_idx in tqdm(range(n_frames), desc="Adding overlay"):
    ret, frame = cap.read()
    if not ret:
        break

    # 1) create the padded frame
    bar       = np.zeros((bar_h, w, 3), dtype=np.uint8)
    new_frame = np.vstack((frame, bar))

    # 2) overlay the velocity text in the bar (left‐aligned)
    vel_text = f"{vel_array[frame_idx-1]:.2f} cm/s"
    (tw, th), _ = cv2.getTextSize(vel_text, font, font_scale, thickness)
    text_x = margin
    text_y = h + margin + th
    cv2.putText(
        new_frame,
        vel_text,
        (text_x, text_y),
        font,
        font_scale,
        (255, 255, 255),
        thickness,
        cv2.LINE_AA
    )

    # 3) draw the dot at the mouse’s X in the center of the bar
    xm = int(np.clip(x_pos[frame_idx-1], 0, w-1))
    y_dot = h + bar_h // 2
    cv2.circle(new_frame, (xm, y_dot), dot_radius, (255, 255, 255), -1)

    out.write(new_frame)

# ── CLEAN UP ────────────────────────────────────────────────────────────────────

cap.release()
out.release()
print("Done – saved with overlay to:", output_vid)

OpenCV: FFMPEG: tag 0x44495658/'XVID' is not supported with codec id 12 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x7634706d/'mp4v'
Adding overlay: 100%|███████████████████| 20570/20570 [00:05<00:00, 3827.94it/s]

Done – saved with overlay to: /Users/johnmarshall/Documents/Analysis/miniscope_analysis/miniscopeLinearTrack/2025.4/m328/2025_04_10_328_15_59_58_b2/My_WebCam/with_velocity_overlay.mp4
